# Image Super Resolution

## Improving ESPCN

### Imports

In [18]:
# Dependencies
import math
import numpy as np
import matplotlib.pyplot as plt
import PIL
from PIL import Image
import cv2
import seaborn as sns
import time
from tqdm import tqdm
import copy
import os
import h5py
import glob

import torch
from torch import nn
import torch.backends.cudnn as cudnn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from torchvision.models import vgg19, vgg13

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.device(0))
print(torch.cuda.get_device_name(0))

print('cuda:0' if torch.cuda.is_available() else 'cpu')

True
1
0
NVIDIA GeForce RTX 4070
cuda:0


### Utils

In [3]:
def calc_patch_size(func):
    def wrapper(args):
        if args.scale == 2:
            args.patch_size = 10
        elif args.scale == 3:
            args.patch_size = 7
        elif args.scale == 4:
            args.patch_size = 6
        else:
            raise Exception('Scale Error', args.scale)
        return func(args)
    return wrapper


def convert_rgb_to_y(img, dim_order='hwc'):
    if dim_order == 'hwc':
        return 16. + (65.738 * img[..., 0] + 129.057 * img[..., 1] + 25.064 * img[..., 2]) / 256.
    else:
        return 16. + (65.738 * img[0] + 129.057 * img[1] + 25.064 * img[2]) / 256.

# These magic numbers were defined by the ITU-R BT.601 standard
# Source: https://en.wikipedia.org/wiki/YCbCr#ITU-R_BT.601_conversion
# Main Wiki page: https://en.wikipedia.org/wiki/Rec._601

# Note: the value '64.738' should have been '65.738', according to Wikipedia. With this change,
# although the model was trained with the wrong coeffiecent of 64.738, the PSNR value slightly
# increases when comparing the final images, however, it decreases when comparing only the
# luminance channel. It might be worth evaluating the model with the fix, to see if PSNR improves.
def convert_rgb_to_ycbcr(img, dim_order='hwc'):
    if dim_order == 'hwc':
        y = 16. + (65.738 * img[..., 0] + 129.057 * img[..., 1] + 25.064 * img[..., 2]) / 256.
        cb = 128. + (-37.945 * img[..., 0] - 74.494 * img[..., 1] + 112.439 * img[..., 2]) / 256.
        cr = 128. + (112.439 * img[..., 0] - 94.154 * img[..., 1] - 18.285 * img[..., 2]) / 256.
    else:
        y = 16. + (65.738 * img[0] + 129.057 * img[1] + 25.064 * img[2]) / 256.
        cb = 128. + (-37.945 * img[0] - 74.494 * img[1] + 112.439 * img[2]) / 256.
        cr = 128. + (112.439 * img[0] - 94.154 * img[1] - 18.285 * img[2]) / 256.
    return np.array([y, cb, cr]).transpose([1, 2, 0])


def convert_ycbcr_to_rgb(img, dim_order='hwc'):
    if dim_order == 'hwc':
        r = 298.082 * img[..., 0] / 256. + 408.583 * img[..., 2] / 256. - 222.921
        g = 298.082 * img[..., 0] / 256. - 100.291 * img[..., 1] / 256. - 208.120 * img[..., 2] / 256. + 135.576
        b = 298.082 * img[..., 0] / 256. + 516.412 * img[..., 1] / 256. - 276.836
    else:
        r = 298.082 * img[0] / 256. + 408.583 * img[2] / 256. - 222.921
        g = 298.082 * img[0] / 256. - 100.291 * img[1] / 256. - 208.120 * img[2] / 256. + 135.576
        b = 298.082 * img[0] / 256. + 516.412 * img[1] / 256. - 276.836
    return np.array([r, g, b]).transpose([1, 2, 0])

def preprocess(img, device):
    img = np.array(img).astype(np.float32)
    ycbcr = convert_rgb_to_ycbcr(img)
    x = ycbcr[..., 0]
    x /= 255.
    x = torch.from_numpy(x).to(device)
    x = x.unsqueeze(0).unsqueeze(0)
    return x, ycbcr

def calc_psnr(img1, img2):
    return 10. * torch.log10(1. / torch.mean((img1 - img2) ** 2))


class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

### Preparation

In [4]:
def train(CFG):
    h5_file = h5py.File(CFG['output_path'], 'w')

    lr_patches = []
    hr_patches = []

    for image_path in sorted(glob.glob('{}/*'.format(CFG['images_dir']))):
        hr = Image.open(image_path).convert('RGB')
        hr_width = (hr.width // CFG['scale']) * CFG['scale']
        hr_height = (hr.height // CFG['scale']) * CFG['scale']
        hr = hr.resize((hr_width, hr_height), resample=Image.BICUBIC)
        lr = hr.resize((hr_width // CFG['scale'], hr_height // CFG['scale']), resample=Image.BICUBIC)
        hr = np.array(hr).astype(np.float32)
        lr = np.array(lr).astype(np.float32)
        hr = convert_rgb_to_y(hr)
        lr = convert_rgb_to_y(lr)

        for i in range(0, lr.shape[0] - CFG['patch_size'] + 1, CFG['stride']):
            for j in range(0, lr.shape[1] - CFG['patch_size'] + 1, CFG['stride']):
                lr_patches.append(lr[i:i + CFG['patch_size'], j:j + CFG['patch_size']])
                hr_patches.append(hr[i * CFG['scale']:i * CFG['scale'] + CFG['patch_size'] * CFG['scale'], j * CFG['scale']:j * CFG['scale'] + CFG['patch_size'] * CFG['scale']])

    lr_patches = np.array(lr_patches)
    hr_patches = np.array(hr_patches)

    h5_file.create_dataset('lr', data=lr_patches)
    h5_file.create_dataset('hr', data=hr_patches)

    h5_file.close()


def eval(CFG):
    h5_file = h5py.File(CFG['output_path'], 'w')

    lr_group = h5_file.create_group('lr')
    hr_group = h5_file.create_group('hr')

    for i, image_path in enumerate(sorted(glob.glob('{}/*'.format(CFG['images_dir'])))):
        hr = Image.open(image_path).convert('RGB')
        hr_width = (hr.width // CFG['scale']) * CFG['scale']
        hr_height = (hr.height // CFG['scale']) * CFG['scale']
        hr = hr.resize((hr_width, hr_height), resample=Image.BICUBIC)
        lr = hr.resize((hr.width // CFG['scale'], hr_height // CFG['scale']), resample=Image.BICUBIC)
        hr = np.array(hr).astype(np.float32)
        lr = np.array(lr).astype(np.float32)
        hr = convert_rgb_to_y(hr)
        lr = convert_rgb_to_y(lr)

        lr_group.create_dataset(str(i), data=lr)
        hr_group.create_dataset(str(i), data=hr)

    h5_file.close()

In [5]:
def train_rgb(CFG):
    h5_file = h5py.File(CFG['output_path'], 'w')

    lr_patches = []
    hr_patches = []

    for image_path in sorted(glob.glob('{}/*'.format(CFG['images_dir']))):
        hr = Image.open(image_path).convert('RGB')
        hr_width = (hr.width // CFG['scale']) * CFG['scale']
        hr_height = (hr.height // CFG['scale']) * CFG['scale']
        hr = hr.resize((hr_width, hr_height), resample=Image.BICUBIC)
        lr = hr.resize((hr_width // CFG['scale'], hr_height // CFG['scale']), resample=Image.BICUBIC)
        hr = np.array(hr).astype(np.float32)
        lr = np.array(lr).astype(np.float32)
        # hr = convert_rgb_to_y(hr)
        # lr = convert_rgb_to_y(lr)

        # print(hr.shape)
        # print(lr.shape)

        for i in range(0, lr.shape[0] - CFG['patch_size'] + 1, CFG['stride']):
            for j in range(0, lr.shape[1] - CFG['patch_size'] + 1, CFG['stride']):
                lr_patches.append(lr[i:i + CFG['patch_size'], j:j + CFG['patch_size']])
                hr_patches.append(hr[i * CFG['scale']:i * CFG['scale'] + CFG['patch_size'] * CFG['scale'], j * CFG['scale']:j * CFG['scale'] + CFG['patch_size'] * CFG['scale']])

    lr_patches = np.array(lr_patches)
    hr_patches = np.array(hr_patches)

    h5_file.create_dataset('lr', data=lr_patches)
    h5_file.create_dataset('hr', data=hr_patches)

    h5_file.close()


def eval_rgb(CFG):
    h5_file = h5py.File(CFG['output_path'], 'w')

    lr_group = h5_file.create_group('lr')
    hr_group = h5_file.create_group('hr')

    for i, image_path in enumerate(sorted(glob.glob('{}/*'.format(CFG['images_dir'])))):
        hr = Image.open(image_path).convert('RGB')
        hr_width = (hr.width // CFG['scale']) * CFG['scale']
        hr_height = (hr.height // CFG['scale']) * CFG['scale']
        hr = hr.resize((hr_width, hr_height), resample=Image.BICUBIC)
        lr = hr.resize((hr.width // CFG['scale'], hr_height // CFG['scale']), resample=Image.BICUBIC)
        hr = np.array(hr).astype(np.float32)
        lr = np.array(lr).astype(np.float32)
        # hr = convert_rgb_to_y(hr)
        # lr = convert_rgb_to_y(lr)

        lr_group.create_dataset(str(i), data=lr)
        hr_group.create_dataset(str(i), data=hr)

    h5_file.close()

In [6]:
CFG_PREPARE_TRAIN = {
    'images_dir': 'datasets/91Image',
    'output_path': 'datasets/HDF5-formatted/91-image-RGB_x3.h5',
    'scale': 3,
    'patch_size': 17,
    'stride': 13
}

train_rgb(CFG_PREPARE_TRAIN)

CFG_PREPARE_EVAL = {
    'images_dir': 'datasets/Set5',
    'output_path': 'datasets/HDF5-formatted/Set5-RGB_x3.h5',
    'scale': 3,
    'patch_size': 17,
    'stride': 13
}

eval_rgb(CFG_PREPARE_EVAL)

### Datasets

In [7]:
# Used for Luminance
class TrainDataset(Dataset):
    def __init__(self, h5_file):
        super(TrainDataset, self).__init__()
        self.h5_file = h5_file

    def __getitem__(self, idx):
        with h5py.File(self.h5_file, 'r') as f:
            return np.expand_dims(f['lr'][idx] / 255., 0), np.expand_dims(f['hr'][idx] / 255., 0)

    def __len__(self):
        with h5py.File(self.h5_file, 'r') as f:
            return len(f['lr'])


class EvalDataset(Dataset):
    def __init__(self, h5_file):
        super(EvalDataset, self).__init__()
        self.h5_file = h5_file

    def __getitem__(self, idx):
        with h5py.File(self.h5_file, 'r') as f:
            return np.expand_dims(f['lr'][str(idx)][:, :] / 255., 0), np.expand_dims(f['hr'][str(idx)][:, :] / 255., 0)

    def __len__(self):
        with h5py.File(self.h5_file, 'r') as f:
            return len(f['lr'])

In [8]:
# Used for RGB
class TrainDatasetRGB(Dataset):
    def __init__(self, h5_file):
        super(TrainDatasetRGB, self).__init__()
        self.h5_file = h5_file

    def __getitem__(self, idx):
        with h5py.File(self.h5_file, 'r') as f:
            return np.transpose(f['lr'][idx] / 255., (2, 0, 1)), np.transpose(f['hr'][idx] / 255., (2, 0, 1))

    def __len__(self):
        with h5py.File(self.h5_file, 'r') as f:
            return len(f['lr'])


class EvalDatasetRGB(Dataset):
    def __init__(self, h5_file):
        super(EvalDatasetRGB, self).__init__()
        self.h5_file = h5_file

    def __getitem__(self, idx):
        with h5py.File(self.h5_file, 'r') as f:
            return np.transpose(f['lr'][str(idx)][:, :] / 255., (2, 0, 1)), np.transpose(f['hr'][str(idx)][:, :] / 255., (2, 0, 1))

    def __len__(self):
        with h5py.File(self.h5_file, 'r') as f:
            return len(f['lr'])

### Model

In [29]:
class ESPCN(nn.Module):
    def __init__(self, scale_factor, num_channels=1):
        super(ESPCN, self).__init__()
        self.first_part = nn.Sequential(
            nn.Conv2d(num_channels, 64, kernel_size=5, padding=5//2),
            nn.Tanh(),
            nn.Conv2d(64, 32, kernel_size=3, padding=3//2),
            nn.Tanh(),
        )
        self.last_part = nn.Sequential(
            nn.Conv2d(32, num_channels * (scale_factor ** 2), kernel_size=3, padding=3 // 2),
            nn.PixelShuffle(scale_factor)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                if m.in_channels == 32:
                    nn.init.normal_(m.weight.data, mean=0.0, std=0.001)
                    nn.init.zeros_(m.bias.data)
                else:
                    nn.init.normal_(m.weight.data, mean=0.0, std=math.sqrt(2/(m.out_channels*m.weight.data[0][0].numel())))
                    nn.init.zeros_(m.bias.data)

    def forward(self, x):
        x = self.first_part(x)
        x = self.last_part(x)
        return x
    
class ESPCNv2(nn.Module):
    def __init__(self, scale_factor, num_channels=1):
        super(ESPCNv2, self).__init__()
        self.first_part = nn.Sequential(
            nn.Conv2d(num_channels, 128, kernel_size=5, padding=5//2),
            nn.Tanh(),
            nn.Conv2d(128, 64, kernel_size=3, padding=3//2),
            nn.Tanh(),
            nn.Conv2d(64, 32, kernel_size=3, padding=3//2),
            nn.Tanh(),
            nn.Conv2d(32, 16, kernel_size=3, padding=3//2),
            nn.Tanh(),
        )
        self.last_part = nn.Sequential(
            nn.Conv2d(16, num_channels * (scale_factor ** 2), kernel_size=3, padding=3 // 2),
            nn.PixelShuffle(scale_factor)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                if m.in_channels == 32:
                    nn.init.normal_(m.weight.data, mean=0.0, std=0.001)
                    nn.init.zeros_(m.bias.data)
                else:
                    nn.init.normal_(m.weight.data, mean=0.0, std=math.sqrt(2/(m.out_channels*m.weight.data[0][0].numel())))
                    nn.init.zeros_(m.bias.data)

    def forward(self, x):
        x = self.first_part(x)
        x = self.last_part(x)
        return x

In [ ]:
def train_espcn(CFG, loss_fn):
    out_dir = os.path.join(CFG['output-dir'], 'x{}'.format(CFG['scale']))

    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    cudnn.benchmark = True
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    torch.manual_seed(CFG['seed'])

    if CFG["type"] == "ESPCN":
        model = ESPCN(scale_factor=CFG['scale'], num_channels=1 if CFG['rgb'] is False else 3).to(device)
    elif CFG["type"] == "ESPCNv2":
        model = ESPCNv2(scale_factor=CFG['scale'], num_channels=1 if CFG['rgb'] is False else 3).to(device)

    optimizer = optim.Adam([
        {'params': model.first_part.parameters()},
        {'params': model.last_part.parameters(), 'lr': CFG['lr'] * 0.1}
    ], lr=CFG['lr'])

    if CFG['rgb'] is False:
        train_dataset = TrainDataset(CFG['train-file'])
    else:
        train_dataset = TrainDatasetRGB(CFG['train-file'])

    train_dataloader = DataLoader(dataset=train_dataset,
                                    batch_size=CFG['batch-size'],
                                    shuffle=True,
                                    # num_workers=CFG['num-workers'],
                                    pin_memory=True)
    
    if CFG['rgb'] is False:
        eval_dataset = EvalDataset(CFG['eval_file'])
    else:
        eval_dataset = EvalDatasetRGB(CFG['eval_file'])

    eval_dataloader = DataLoader(dataset=eval_dataset, batch_size=1)

    best_weights = copy.deepcopy(model.state_dict())
    best_epoch = 0
    best_psnr = 0.0

    for epoch in range(CFG['num-epochs']):
        for param_group in optimizer.param_groups:
            param_group['lr'] = CFG['lr'] * (0.1 ** (epoch // int(CFG['num-epochs'] * 0.8)))

        model.train()
        epoch_losses = AverageMeter()

        with tqdm(total=(len(train_dataset) - len(train_dataset) % CFG['batch-size']), ncols=80) as t:
            t.set_description('epoch: {}/{}'.format(epoch, CFG['num-epochs'] - 1))

            for data in train_dataloader:
                inputs, labels = data

                inputs = inputs.to(device)
                labels = labels.to(device)

                # print(inputs.shape)

                preds = model(inputs)

                loss = loss_fn(preds, labels)

                epoch_losses.update(loss.item(), len(inputs))

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                t.set_postfix(loss='{:.6f}'.format(epoch_losses.avg))
                t.update(len(inputs))

        torch.save(model.state_dict(), os.path.join(out_dir, 'epoch_{}.pth'.format(epoch)))

        model.eval()
        epoch_psnr = AverageMeter()

        for data in eval_dataloader:
            inputs, labels = data

            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.no_grad():
                preds = model(inputs).clamp(0.0, 1.0)

            epoch_psnr.update(calc_psnr(preds, labels), len(inputs))

        print('eval psnr: {:.2f}'.format(epoch_psnr.avg))

        if epoch_psnr.avg > best_psnr:
            best_epoch = epoch
            best_psnr = epoch_psnr.avg
            best_weights = copy.deepcopy(model.state_dict())

    print('best epoch: {}, psnr: {:.2f}'.format(best_epoch, best_psnr))
    torch.save(best_weights, os.path.join(out_dir, 'best.pth'))

In [ ]:
CFG_DEFAULT = {
    'output-dir': 'out/default',
    'train-file': 'datasets/HDF5-formatted/91-image_x3.h5',
    'eval_file': 'datasets/HDF5-formatted/Set5_x3.h5',
    'scale': 3,
    'lr': 1e-3,
    'batch-size': 16,
    'num-epochs': 200,
    'num-workers': 8,
    'seed': 123,
    'rgb': False,
    'type': "ESPCN"
}

train_espcn(CFG_DEFAULT, nn.MSELoss())

In [ ]:
CFG_DEFAULT_SMALL_LEARNING_RATE = {
    'output-dir': 'out/default',
    'train-file': 'datasets/HDF5-formatted/91-image_x3.h5',
    'eval_file': 'datasets/HDF5-formatted/Set5_x3.h5',
    'scale': 3,
    'lr': 1e-4,
    'batch-size': 16,
    'num-epochs': 200,
    'num-workers': 8,
    'seed': 123,
    'rgb': False,
    'type': "ESPCN"
}

train_espcn(CFG_DEFAULT_SMALL_LEARNING_RATE, nn.MSELoss())

In [ ]:
CFG_DEFAULT_RGB = {
    'output-dir': 'out/default-rgb',
    'train-file': 'datasets/HDF5-formatted/91-image-RGB_x3.h5',
    'eval_file': 'datasets/HDF5-formatted/Set5-RGB_x3.h5',
    'scale': 3,
    'lr': 1e-3,
    'batch-size': 16,
    'num-epochs': 200,
    'num-workers': 8,
    'seed': 123,
    'rgb': True,
    'type': "ESPCN"
}

train_espcn(CFG_DEFAULT_RGB, nn.MSELoss())

In [20]:
class CharbonnierLoss(nn.Module):
    """Charbonnier Loss (Robust L1)"""
    def __init__(self, eps=1e-3):
        super(CharbonnierLoss, self).__init__()
        self.eps = eps

    def forward(self, x, y):
        diff = x - y
        loss = torch.sqrt(diff * diff + self.eps * self.eps)

        return torch.mean(loss)

class PerceptualLossVGG19(nn.Module):
    def __init__(self):
        super(PerceptualLossVGG19, self).__init__()

        self.l1Loss = CharbonnierLoss()

        vgg = vgg19(pretrained=True).features[:36].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg

        self.criterion = nn.MSELoss()

    def forward(self, sr, hr):
        pixelLoss = self.l1Loss(sr, hr)
        perceptualLoss = self.criterion(self.vgg(sr), self.vgg(hr))

        return pixelLoss + 0.01 * perceptualLoss

In [ ]:
CFG_IMPROVED = {
    'output-dir': 'out/improved',
    'train-file': 'datasets/HDF5-formatted/91-image-RGB_x3.h5',
    'eval_file': 'datasets/HDF5-formatted/Set5-RGB_x3.h5',
    'scale': 3,
    'lr': 1e-3,
    'batch-size': 16,
    'num-epochs': 200,
    'num-workers': 8,
    'seed': 123,
    'rgb': True,
    'type': "ESPCN"
}
    
cudnn.benchmark = True
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

perceptualLoss = PerceptualLossVGG19().to(device)

train_espcn(CFG_IMPROVED, perceptualLoss)

In [ ]:
CFG_PERCEPTUAL_MANY_EPOCHS = {
    'output-dir': 'out/perceptual-500epochs',
    'train-file': 'datasets/HDF5-formatted/91-image-RGB_x3.h5',
    'eval_file': 'datasets/HDF5-formatted/Set5-RGB_x3.h5',
    'scale': 3,
    'lr': 1e-3,
    'batch-size': 16,
    'num-epochs': 500,
    'num-workers': 8,
    'seed': 123,
    'rgb': True,
    'type': "ESPCN"
}
    
cudnn.benchmark = True
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

perceptualLoss = PerceptualLossVGG19().to(device)

train_espcn(CFG_PERCEPTUAL_MANY_EPOCHS, perceptualLoss)

In [ ]:
CFG_ESPCN_V2 = {
    'output-dir': 'out/espcn-v2',
    'train-file': 'datasets/HDF5-formatted/91-image_x3.h5',
    'eval_file': 'datasets/HDF5-formatted/Set5_x3.h5',
    'scale': 3,
    'lr': 1e-3,
    'batch-size': 16,
    'num-epochs': 200,
    'num-workers': 8,
    'seed': 123,
    'rgb': False,
    'type': "ESPCNv2"
}

train_espcn(CFG_ESPCN_V2, nn.MSELoss())

### Test performance

In [41]:
def eval_espcn_image(espcn, image_path: str, output_path: str, scale: int, device):
    image = Image.open(image_path).convert('RGB')

    image_width = (image.width // scale) * scale
    image_height = (image.height // scale) * scale

    hr = image.resize((image_width, image_height), resample=Image.BICUBIC)
    lr = hr.resize((hr.width // scale, hr.height // scale), resample=Image.BICUBIC)
    bicubic = lr.resize((lr.width * scale, lr.height * scale), resample=Image.BICUBIC)
    # bicubic.save(args.image_file.replace('.', '_bicubic_x{}.'.format(scale)))

    lr, _ = preprocess(lr, device)
    hr, _ = preprocess(hr, device)
    _, ycbcr = preprocess(bicubic, device)

    with torch.no_grad():
        preds = espcn(lr).clamp(0.0, 1.0)

    psnr = calc_psnr(hr, preds)
    print('PSNR: {:.2f}'.format(psnr))

    preds = preds.mul(255.0).cpu().numpy().squeeze(0).squeeze(0)

    output = np.array([preds, ycbcr[..., 1], ycbcr[..., 2]]).transpose([1, 2, 0])
    output = np.clip(convert_ycbcr_to_rgb(output), 0.0, 255.0).astype(np.uint8)
    output = Image.fromarray(output)
    output.save(output_path)

    return psnr

def eval_espcn(weights_path: str, suffix: str, type="ESPCN"):
    CFG_TEST = {
        'scale': 3
    }

    cudnn.benchmark = True
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    if type == "ESPCN":
        model = ESPCN(scale_factor=CFG_TEST['scale']).to(device)
    elif type == "ESPCNv2":
        model = ESPCNv2(scale_factor=CFG_TEST['scale']).to(device)

    state_dict = model.state_dict()
    for n, p in torch.load(weights_path, map_location=lambda storage, loc: storage).items():
        if n in state_dict.keys():
            state_dict[n].copy_(p)
        else:
            raise KeyError(n)

    model.eval()

    psnr_total = 0

    psnr_total += eval_espcn_image(model, 'datasets/Set5/baby.png', 'datasets/Set5/baby_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image(model, 'datasets/Set5/bird.png', 'datasets/Set5/bird_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image(model, 'datasets/Set5/butterfly.png', 'datasets/Set5/butterfly_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image(model, 'datasets/Set5/head.png', 'datasets/Set5/head_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image(model, 'datasets/Set5/woman.png', 'datasets/Set5/woman_{}.png'.format(suffix), CFG_TEST['scale'], device)

    psnr_avg = psnr_total / 5

    print("Average: {:.2f}".format(psnr_avg))

    return psnr_avg


In [42]:
eval_espcn('pretrained-weights/ESPCN/espcn_x3.pth', suffix="espcn_x3_github")
print()

eval_espcn('pretrained-weights/ESPCN/Trained/best_MSE.pth', suffix="espcn_x3")
print()

eval_espcn('pretrained-weights/ESPCN/Trained/latest_MSE_v2.pth', suffix="espcn_v2_x3", type="ESPCNv2")
print()

PSNR: 35.98
PSNR: 34.74
PSNR: 27.61
PSNR: 35.04
PSNR: 30.84
Average: 32.84

PSNR: 36.00
PSNR: 34.72
PSNR: 27.51
PSNR: 35.02
PSNR: 30.84
Average: 32.82

PSNR: 35.97
PSNR: 34.84
PSNR: 27.72
PSNR: 34.96
PSNR: 30.99
Average: 32.90



#### RGB Eval

In [14]:
def img_preprocess(img, device):
    img = np.array(img).astype(np.float32)
    img /= 255.
    img = torch.from_numpy(img).to(device)

    return img.permute((2, 0, 1))

def eval_espcn_image_rgb(espcn, image_path: str, out_path: str, scale: int, device):
    image = Image.open(image_path).convert('RGB')

    image_width = (image.width // scale) * scale
    image_height = (image.height // scale) * scale

    hr = image.resize((image_width, image_height), resample=Image.BICUBIC)
    lr = hr.resize((hr.width // scale, hr.height // scale), resample=Image.BICUBIC)
    bicubic = lr.resize((lr.width * scale, lr.height * scale), resample=Image.BICUBIC)
    # bicubic.save(args.image_file.replace('.', '_bicubic_x{}.'.format(scale)))

    lr = img_preprocess(lr, device) # torch.permute(lr / 255., (2, 0, 1))
    hr = img_preprocess(hr, device) # torch.permute(hr / 255., (2, 0, 1))
    # _, ycbcr = preprocess(bicubic, device)

    with torch.no_grad():
        preds = espcn(lr).clamp(0.0, 1.0)

    psnr = calc_psnr(hr, preds)
    print('PSNR: {:.2f}'.format(psnr))

    output = preds.permute((1, 2, 0)).mul(255.0).cpu().numpy()
    output = np.clip(output, 0.0, 255.0).astype(np.uint8)
    output = Image.fromarray(output)
    output.save(out_path) # image_path.replace('.', '_espcn_rgb_x{}.'.format(scale)))

    return psnr

def eval_espcn_rgb(weights_path: str, suffix: str):
    CFG_TEST = {
        'scale': 3
    }

    cudnn.benchmark = True
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    model = ESPCN(scale_factor=CFG_TEST['scale'], num_channels=3).to(device)

    state_dict = model.state_dict()
    for n, p in torch.load(weights_path, map_location=lambda storage, loc: storage).items():
        if n in state_dict.keys():
            state_dict[n].copy_(p)
        else:
            raise KeyError(n)

    model.eval()

    psnr_total = 0

    psnr_total += eval_espcn_image_rgb(model, 'datasets/Set5/baby.png', 'datasets/Set5/baby_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image_rgb(model, 'datasets/Set5/bird.png', 'datasets/Set5/bird_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image_rgb(model, 'datasets/Set5/butterfly.png', 'datasets/Set5/butterfly_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image_rgb(model, 'datasets/Set5/head.png', 'datasets/Set5/head_{}.png'.format(suffix), CFG_TEST['scale'], device)
    psnr_total += eval_espcn_image_rgb(model, 'datasets/Set5/woman.png', 'datasets/Set5/woman_{}.png'.format(suffix), CFG_TEST['scale'], device)

    psnr_avg = psnr_total / 5

    print("Average: {:.2f}".format(psnr_avg))

    return psnr_avg


In [40]:
eval_espcn_rgb('pretrained-weights/ESPCN/Trained/best_MSE_RGB.pth', suffix='espcn_rgb_x3')
print()

eval_espcn_rgb('pretrained-weights/ESPCN/Trained/latest_VGG_RGB.pth', suffix='espcn_rgb_trash_x3')
print()

eval_espcn_rgb('pretrained-weights/ESPCN/Trained/latest_VGG_L1_RGB.pth', suffix='espcn_rgb_trash_x3')
print()

eval_espcn_rgb('pretrained-weights/ESPCN/Trained/latest_VGG_L1_500epochs_RGB.pth', suffix='espcn_rgb_500epochs_x3')
print()

PSNR: 34.42
PSNR: 32.75
PSNR: 26.41
PSNR: 32.24
PSNR: 29.54
Average: 31.07

PSNR: 19.41
PSNR: 18.34
PSNR: 14.61
PSNR: 20.33
PSNR: 17.14
Average: 17.97

PSNR: 33.57
PSNR: 31.59
PSNR: 24.63
PSNR: 31.79
PSNR: 28.61
Average: 30.04

PSNR: 34.49
PSNR: 32.72
PSNR: 26.20
PSNR: 32.25
PSNR: 29.60
Average: 31.05

